#  Heart Failure Prediction — Prediction Model Pipeline

**Dataset:** Heart Failure Clinical Records Dataset  
**Task:** Binary Classification — Predict `DEATH_EVENT` (0 = Survived, 1 = Died)  
**Models:** SVM, Random Forest, Gradient Boosting  

---

## 1. Install & Import Libraries

In [ ]:
# Install if needed (Colab)
# !pip install scikit-learn pandas numpy matplotlib seaborn

import os
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110

print('All libraries loaded successfully ')

## 2. Load Dataset

In [ ]:
# Adjust path as needed
DATA_PATH = '../heart_failure_clinical_records_dataset.csv'

df = pd.read_csv(DATA_PATH)

print(f'Dataset shape: {df.shape}')
print(f'\nColumns ({len(df.columns)}):')
print(df.columns.tolist())
df.head(10)

In [ ]:
print('Data Types:')
print(df.dtypes)
print('\nBasic Info:')
df.info()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print('Statistical Summary:')
df.describe().round(2)

In [ ]:
print('Missing Values per Column:')
missing = df.isnull().sum()
print(missing[missing >= 0])
print(f'\nTotal missing: {df.isnull().sum().sum()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
counts = df['DEATH_EVENT'].value_counts()
axes[0].bar(['Survived (0)', 'Died (1)'], counts.values,
            color=['#48bb78', '#e53e3e'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=['Survived', 'Died'],
            colors=['#48bb78', '#e53e3e'], autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 11})
axes[1].set_title('Class Proportion', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()
print(f'Class imbalance ratio: {counts[0]/counts[1]:.2f}:1')

In [ ]:
FEATURES = [
    'age', 'anaemia', 'creatinine_phosphokinase', 'diabetes',
    'ejection_fraction', 'high_blood_pressure', 'platelets',
    'serum_creatinine', 'serum_sodium', 'sex', 'smoking', 'time'
]

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

for i, col in enumerate(FEATURES):
    survived = df[df['DEATH_EVENT'] == 0][col]
    died     = df[df['DEATH_EVENT'] == 1][col]
    axes[i].hist(survived, alpha=0.6, bins=20, color='#48bb78', label='Survived')
    axes[i].hist(died,     alpha=0.6, bins=20, color='#e53e3e', label='Died')
    axes[i].set_title(col.replace('_', ' ').title(), fontsize=10, fontweight='bold')
    axes[i].legend(fontsize=8)

plt.suptitle('Feature Distributions by Outcome', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
numerical = ['age', 'creatinine_phosphokinase', 'ejection_fraction',
             'platelets', 'serum_creatinine', 'serum_sodium', 'time']

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i, col in enumerate(numerical):
    df.boxplot(column=col, by='DEATH_EVENT', ax=axes[i],
               boxprops=dict(color='#2d3748'),
               medianprops=dict(color='#e53e3e', linewidth=2))
    axes[i].set_title(col.replace('_', ' ').title(), fontsize=10, fontweight='bold')
    axes[i].set_xlabel('DEATH_EVENT (0=Survived, 1=Died)')

axes[-1].set_visible(False)
plt.suptitle('Box Plots: Features vs Outcome', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(13, 9))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn_r', center=0, vmin=-1, vmax=1,
            linewidths=.5, annot_kws={'size': 8},
            cbar_kws={'label': 'Correlation Coefficient'})
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=16)
plt.tight_layout()
plt.show()

print('\nTop correlations with DEATH_EVENT:')
print(corr['DEATH_EVENT'].drop('DEATH_EVENT').abs().sort_values(ascending=False))

## 4. Preprocessing Pipeline

In [ ]:
X = df[FEATURES].values
y = df['DEATH_EVENT'].values

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'Positive class ratio: {y.mean():.3f}')

# Train/Test Split — stratified to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'\nTrain size: {len(X_train)} | Test size: {len(X_test)}')
print(f'Train positives: {y_train.mean():.3f} | Test positives: {y_test.mean():.3f}')

# Standard Scaling
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'\nScaled X_train — mean: {X_train_sc.mean():.4f}, std: {X_train_sc.std():.4f}')
print('Preprocessing complete ')

## 5. Model Training

In [ ]:
models = {
    'SVM (RBF Kernel)': SVC(kernel='rbf', probability=True, random_state=42),
    'Random Forest':    RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train_sc, y_train)
    print(f' {name} trained')
    results[name] = model

print('\nAll models trained!')

## 6. Model Evaluation

In [ ]:
evaluation = {}

for name, model in results.items():
    y_pred = model.predict(X_test_sc)
    y_prob = model.predict_proba(X_test_sc)[:, 1]

    evaluation[name] = {
        'model':     model,
        'y_pred':    y_pred,
        'y_prob':    y_prob,
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall':    recall_score(y_test, y_pred),
        'f1':        f1_score(y_test, y_pred),
        'roc_auc':   roc_auc_score(y_test, y_prob),
        'cv_acc':    cross_val_score(model, X_train_sc, y_train, cv=5, scoring='accuracy').mean(),
    }

    print(f'\n{"="*50}')
    print(f'  {name}')
    print(f'{"="*50}')
    print(f'  Accuracy:    {evaluation[name]["accuracy"]:.4f}')
    print(f'  Precision:   {evaluation[name]["precision"]:.4f}')
    print(f'  Recall:      {evaluation[name]["recall"]:.4f}')
    print(f'  F1-Score:    {evaluation[name]["f1"]:.4f}')
    print(f'  ROC-AUC:     {evaluation[name]["roc_auc"]:.4f}')
    print(f'  CV Accuracy: {evaluation[name]["cv_acc"]:.4f}')
    print(f'\n  Classification Report:')
    print(classification_report(y_test, y_pred, target_names=['Survived', 'Died']))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (name, ev) in zip(axes, evaluation.items()):
    cm = confusion_matrix(y_test, ev['y_pred'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['Survived', 'Died'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(name, fontsize=11, fontweight='bold')

plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))

colors = ['#e53e3e', '#4299e1', '#48bb78']
for (name, ev), color in zip(evaluation.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, ev['y_prob'])
    plt.plot(fpr, tpr, color=color, linewidth=2,
             label=f"{name} (AUC={ev['roc_auc']:.3f})")

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
plt.xlabel('False Positive Rate', fontsize=11)
plt.ylabel('True Positive Rate', fontsize=11)
plt.title('ROC Curves — All Models', fontsize=13, fontweight='bold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'cv_acc']
metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'CV Acc']

model_names = list(evaluation.keys())
x = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 6))

colors = ['#e53e3e', '#4299e1', '#48bb78']
for i, (name, color) in enumerate(zip(model_names, colors)):
    vals = [evaluation[name][m] for m in metrics]
    bars = ax.bar(x + i*width, vals, width, label=name, color=color, alpha=0.85, edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=7.5, fontweight='bold')

ax.set_xticks(x + width)
ax.set_xticklabels(metric_labels, fontsize=10)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Model Performance Comparison', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

# Summary table
summary = pd.DataFrame({
    name: {label: evaluation[name][m] for label, m in zip(metric_labels, metrics)}
    for name in model_names
}).T.round(4)
print('\nSummary Table:')
print(summary)

## 7. Best Model Selection & Hyperparameter Tuning

In [ ]:
best_name = max(evaluation, key=lambda n: evaluation[n]['accuracy'])
print(f'Best model by accuracy: {best_name}')
print(f'  Accuracy: {evaluation[best_name]["accuracy"]:.4f}')
print(f'  ROC-AUC:  {evaluation[best_name]["roc_auc"]:.4f}')

In [ ]:
if best_name == 'Random Forest':
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5],
    }
    base_clf = RandomForestClassifier(random_state=42)
elif best_name == 'Gradient Boosting':
    param_grid = {
        'n_estimators': [100, 200],
        'learning_rate': [0.05, 0.1],
        'max_depth': [3, 5],
    }
    base_clf = GradientBoostingClassifier(random_state=42)
else:
    param_grid = {
        'C': [0.1, 1, 10],
        'gamma': ['scale', 'auto'],
    }
    base_clf = SVC(kernel='rbf', probability=True, random_state=42)

print(f'Running GridSearchCV for {best_name}...')
grid = GridSearchCV(base_clf, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid.fit(X_train_sc, y_train)

tuned_model = grid.best_estimator_
tuned_acc = accuracy_score(y_test, tuned_model.predict(X_test_sc))

print(f'\nBest params: {grid.best_params_}')
print(f'CV best score: {grid.best_score_:.4f}')
print(f'Test accuracy (tuned): {tuned_acc:.4f}')

final_model = tuned_model
print(f'\nFinal model: {best_name} (tuned) — Accuracy={tuned_acc:.4f}')

In [ ]:
if hasattr(final_model, 'feature_importances_'):
    importances = final_model.feature_importances_
    feat_df = pd.DataFrame({'Feature': FEATURES, 'Importance': importances})
    feat_df = feat_df.sort_values('Importance', ascending=True)

    plt.figure(figsize=(10, 6))
    colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(FEATURES)))
    plt.barh(feat_df['Feature'], feat_df['Importance'], color=colors)
    plt.xlabel('Feature Importance', fontsize=11)
    plt.title(f'Feature Importances — {best_name}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print('Feature Importances:')
    print(feat_df.sort_values('Importance', ascending=False).to_string(index=False))
else:
    print(f'{best_name} does not expose feature importances directly.')

## 8. Save Model, Scaler & Feature Names

In [ ]:
ML_DIR = '../app/ml'
os.makedirs(ML_DIR, exist_ok=True)

model_path   = os.path.join(ML_DIR, 'best_model.pkl')
scaler_path  = os.path.join(ML_DIR, 'scaler.pkl')
feature_path = os.path.join(ML_DIR, 'feature_names.pkl')

with open(model_path,   'wb') as f: pickle.dump(final_model, f)
with open(scaler_path,  'wb') as f: pickle.dump(scaler, f)
with open(feature_path, 'wb') as f: pickle.dump(FEATURES, f)

print(f' Model saved   → {model_path}')
print(f' Scaler saved  → {scaler_path}')
print(f' Features saved → {feature_path}')

In [ ]:
# Verify saved model
with open(model_path, 'rb') as f:
    loaded_model = pickle.load(f)
with open(scaler_path, 'rb') as f:
    loaded_scaler = pickle.load(f)

verify_acc = accuracy_score(y_test, loaded_model.predict(loaded_scaler.transform(X_test)))
print(f'Loaded model verification accuracy: {verify_acc:.4f}')
print('Model verification complete ')

# Sample prediction
sample = np.array([[75, 0, 582, 0, 20, 1, 265000, 1.9, 130, 1, 0, 4]])
sample_sc = loaded_scaler.transform(sample)
pred = loaded_model.predict(sample_sc)[0]
prob = loaded_model.predict_proba(sample_sc)[0][1]
print(f'\nSample prediction: {"Died" if pred == 1 else "Survived"} (probability={prob:.3f})')

##  Pipeline Complete

| Step | Status |
|------|--------|
| Data Loading |  |
| EDA |  |
| Preprocessing & Scaling |  |
| SVM Training |  |
| Random Forest Training |  |
| Gradient Boosting Training |  |
| Evaluation & Comparison |  |
| Hyperparameter Tuning |  |
| Model Saving |  |